In [12]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup, Comment
import re
import time

## Version 1.1 
* Offensive/defensive ratings
* Injuries
* Home court advantage (constant)

## Todo
* minimize requests (combine injuries with https://www.basketball-reference.com/teams/%s/2024.html call)
* add features (historical_performances, fatigue, see more below...)
* optimize weights using some algorithm (gradient descent)
* automize picking (using voice assistant or making drag and drop)
* more advanced shit like context of player's role,  
* EDA of all features

In [13]:
conversion_table = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "CHO": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "BRK": "Brooklyn Nets",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHO": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
    "SAS": "San Antonio Spurs",
}

### Version 1.1: using pure linear weights to calculate odds

Offensive/defensive ratings //
Injuries //
Historical performances //
Home court advantage //
Fatigue (recency of games) //
Possibly momentum? //
Sentiment analysis? //
Advanced stats?

With each of these features, assign a weight of importance to each feature, then pull the real time data and calculate the total score



In [14]:
weights = [
    0.4,
    0.4,
    0.2,
    None,
    None,
    None,
    None,
    None,
]  # offdef_ratings, injuries, homecourt_advantage, historical_performances, fatigue, momentum, sentiment, advanced

For injuries, assign a value of importance for each player on a team. Give each team a overall value of 1, and if a player is out, then subtract from the overall value. Each player's value will be normalized so that the sum of all players' values add up to 1.

In [15]:
def normalize(data):
    total = sum(data)
    normalized_data = [x / total for x in data]
    return normalized_data

### Functions to collect and calculate data

In [16]:
def calculate_player_weights(df):
    """
    df (pd.DataFrame) : dataframe of basketballreference html
    return (dict): player --> sum of weights
    """
    player_weights = [
        0.2,
        0.2,
        0.2,
        0.2,
        0.2,
    ]  # points, assists, rebounds, win shares, value over replacement player

    player_scores = {}

    for index, player in df[1].iterrows():  # pts, ast, rebs
        name = player["Player"]
        player_scores[name] = []
        player_scores[name].append(player["PTS"])
        player_scores[name].append(player["AST"])
        player_scores[name].append(player["TRB"])

    for index, player in df[3].iterrows():
        name = player["Player"]
        player_scores[name].append(player["BPM"])
        player_scores[name].append(player["VORP"])

    scores = []
    for player in player_scores:
        for i in range(len(player_scores[player])):
            player_scores[player][i] = player_scores[player][i] * player_weights[i]
        player_scores[player] = sum(player_scores[player])
        scores.append(player_scores[player])
    scores = normalize(scores)
    for i, player in enumerate(player_scores):
        player_scores[player] = scores[i]

    return player_scores

In [17]:
def calculate_team_score(player_scores, injuries):
    """
    players_scores (dict): player: value
    injuries (list) : [[out players], [day to day players]]
    return (float): team score with, team score without
    """

    team_score_with = 1 # consider day to day players as out
    team_score_without = 1 # don't consider day to day players as out 

    for lst in injuries:
        for player in lst:
            if player not in player_scores: # for whatever reason, player doesnt show up on roster/injury report 
                pass
            else:
                score = player_scores[player]
                team_score_with -= score

    for player in injuries[0]:
        if player not in player_scores:
            pass
        else:
            score = player_scores[player]
            team_score_without -= score

    if len(injuries[1]) == 0:
        return team_score_with, team_score_without
    else:
        print("Day to Day Players (check for most recent update): ", injuries[1])
        return team_score_with, team_score_without

In [18]:
def find_injuries():
    """
    return (dict): team: [[out players], [day to day players]]
    """
    df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")
    injuries = {}
    for index, player in df[0].iterrows():
        if player["Team"] not in injuries:
            if "Day To Day" in player["Description"].split("-")[0]:
                injuries[player["Team"]] = [[], [player["Player"]]]
            else:
                injuries[player["Team"]] = [[player["Player"]], []]
        else:
            if "Day To Day" in player["Description"].split("-")[0]: 
                injuries[player["Team"]][1].append(player['Player'])
            else: # player is out 
                injuries[player["Team"]][0].append(player['Player'])
    return injuries

In [19]:
def find_offdef_ratings(team_abbr):
    """
    team_abbr (str) : team abbreivation (CHI, BOS, LAL)
    return (list[float]) : [off_rtg, def_rtg]
    """
    constant = 200 # for inverse relationship for defensive ratings 
    url = "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")
    tag = soup.find("div", {"id": "all_team_misc"})

    for element in tag(text=lambda text: isinstance(text, Comment)):
        soup = BeautifulSoup(element, "html.parser")
    
    rtgs = soup.find_all("td", {"data-stat": ["off_rtg", "def_rtg"]})
    rtgs = [float(rtgs[0].get_text()), constant - float(rtgs[1].get_text())]
    return rtgs

In [20]:
def pipeline(team_abbr):
    """
    team_abbr (str): team abbreviation (BOS, CHI, LAL)
    is_home (boolean) : is the team home
    return (list): [offdef_ratings, injuries, historical_performances, fatigue, homecourt_advantage]
    """
    url = "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
    df = pd.read_html(url)  # call 1
    time.sleep(5)
    offdef_rtgs = find_offdef_ratings(team_abbr)  # call 2
    time.sleep(5)
    injuries = find_injuries()  # call 3
    player_weights = calculate_player_weights(df)
    if conversion_table[team_abbr] in injuries:
        team_score_with, team_score_without = calculate_team_score(
            player_weights, injuries[conversion_table[team_abbr]]
        )
    else:
        team_score_with, team_score_without = calculate_team_score(
            player_weights, [[], []]
        )

    data = [sum(offdef_rtgs), team_abbr, team_score_with, team_score_without]

    return data

In [21]:
def compare_teams(team1, team2):
    """
    team1 (list): [offdef_ratings, team_abbr, team_score_with, team_score_without] *home team
    team2 (list): [offdef_ratings, team_abbr, team_score_with, team_score_without] *away team
    return (str): team winner abbreviation
    """

    include = True

    team1_sum = 0
    team2_sum = 0

    for i in range(1):
        team1[i] = team1[i] * weights[i]
        team1_sum += team1[i]
    for i in range(1):
        team2[i] = team2[i] * weights[i]
        team2_sum += team2[i]

    team1_sum += 0.2  # home court advantage

    if include:
        team1_sum += team1[-2]
        team2_sum += team2[-2]
        print(
            str(team1[-3])
            + " - sum with: "
            + str(team1_sum)
            + " // sum without: "
            + str(team1_sum - team1[-2] + team1[-1])
            + " // totals: "
            + str(team1)
        )
        print(
            str(team2[-3])
            + " - sum with: "
            + str(team2_sum)
            + " // sum without: "
            + str(team2_sum - team2[-2] + team2[-1])
            + " // totals: "
            + str(team2)
        )
    else:
        team1_sum += team1[-1]
        team2_sum += team2[-1]
        print(
            str(team1[-3])
            + " - sum with: "
            + str(team1_sum - team1[-1] + team1[-2])
            + " // sum without: "
            + str(team1_sum)
            + " // totals: "
            + str(team1)
        )
        print(
            str(team2[-3])
            + " - sum with: "
            + str(team2_sum - team2[-1] + team2[-2])
            + " // sum without: "
            + str(team2_sum)
            + " // totals: "
            + str(team2)
        )

    if team1_sum > team2_sum:
        difference = team1_sum - team2_sum
        return team1[-3], difference
    else:
        difference = team2_sum - team1_sum
        return team2[-3], difference

### Outcomes

In [22]:
compare_teams(pipeline("PHO"), pipeline("GSW"))

PHO - sum with: 81.72485342019546 // sum without: 81.72485342019546 // totals: [80.84000000000002, 'PHO', 0.6848534201954397, 0.6848534201954397]
GSW - sum with: 81.14325481798716 // sum without: 81.14325481798716 // totals: [80.2, 'GSW', 0.943254817987152, 0.943254817987152]


('PHO', 0.5815986022083024)

In [ ]:
compare_teams(pipeline("ORL"), pipeline("CLE"))

Day to Day Players (check for most recent update):  ['Jalen Suggs']
ORL - sum with: 0.29786019971469335 sum without: 0.8378506894912031 totals: [81.96000000000001, 0.29786019971469335, 0.8378506894912031, 'ORL']
CLE - sum with: 0.30688221709006924 sum without: None totals: [80.28, 0.30688221709006924, None, 'CLE']


('ORL', 1.8709779826246375)

In [ ]:
compare_teams(pipeline("DET"), pipeline("IND"))

Day to Day Players:  ['Marvin Bagley III']
Day to Day Players:  ['Aaron Nesmith']
[75.88, 0.31777434312210207, 0.2]
[81.36, 0.3183033656062702]
76.3977743431221
81.67830336560627


False

In [ ]:
compare_teams(pipeline("CHO"), pipeline("MIA"))

Day to Day Players (check for most recent update):  ['Nick Smith Jr.', 'Mark Williams']
Day to Day Players (check for most recent update):  ['Kyle Lowry']
CHO - sum with: 0.28143426294820717 sum without: 0.8298804780876494 totals: [76.84000000000002, 0.28143426294820717, 0.8298804780876494, 'CHO']
MIA - sum with: 0.2662983425414365 sum without: 0.739542225730071 totals: [80.36000000000001, 0.2662983425414365, 0.739542225730071, 'MIA']


('MIA', 3.304864079593216)

In [ ]:
compare_teams(pipeline("ATL"), pipeline("DEN"))

Day to Day Players:  ["De'Andre Hunter"]
Day to Day Players:  ['Jamal Murray']
[79.92000000000002, 0.38530035335689045, 0.2]
[81.2, 0.34729981378026076]
80.50530035335692
81.54729981378027


False

In [ ]:
compare_teams(pipeline("NYK"), pipeline("TOR"))

Day to Day Players:  ['Jalen Brunson', 'Immanuel Quickley']
NYK - sum with: 0.15609756097560973 sum without: 0.8628048780487805
TOR - sum with: 0.3972073921971253 sum without: None


'NYK'

In [ ]:
compare_teams(pipeline("HOU"), pipeline("SAS"))

Day to Day Players:  ['Reggie Bullock', 'Tari Eason']
[81.4, 0.37245901639344264, 0.2]
[75.32000000000001, 0.4]
81.97245901639346
75.72000000000001


True

In [ ]:
compare_teams(pipeline("MEM"), pipeline("DAL"))

Day to Day Players:  ['Jake LaRavia']
[77.64, 0.34531704479348463, 0.2]
[81.28, 0.26338329764453955]
78.18531704479349
81.54338329764454


False

In [ ]:
compare_teams(pipeline("MIL"), pipeline("CHI"))

Day to Day Players:  ['Chris Livingston']
Day to Day Players:  ['Alex Caruso', 'Patrick Williams']
[81.36, 0.30334572490706324, 0.2]
[78.32000000000001, 0.23926701570680625]
81.86334572490706
78.55926701570681


True

In [ ]:
compare_teams(pipeline("NOP"), pipeline("MIN"))

Day to Day Players:  ['Anthony Edwards', 'Jaden McDaniels', 'Jordan McLaughlin']
[79.68, 0.3713892709766162, 0.2]
[83.16000000000001, 0.272651356993737]
80.25138927097663
83.43265135699374


False

In [ ]:
compare_teams(pipeline("OKC"), pipeline("UTA"))

Day to Day Players:  ['John Collins']
OKC - sum with: 0.37320481927710847 sum without: None
UTA - sum with: 0.20185185185185184 sum without: 0.6448412698412698


'OKC'

In [ ]:
compare_teams(pipeline("SAC"), pipeline("BRK"))

Day to Day Players:  ['Colby Jones', 'Malik Monk']
[79.88, 0.32941176470588235, 0.2]
[81.52000000000001, 0.2975763962065332]
80.40941176470588
81.81757639620655


False

In [ ]:
compare_teams(pipeline("LAC"), pipeline("POR"))

Day to Day Players:  ['Deandre Ayton', 'Malcolm Brogdon']
[81.64000000000001, 0.3658666666666667, 0.2]
[77.24000000000001, 0.20492610837438427]
82.20586666666668
77.44492610837439


True

### Code Testing

In [ ]:
calculate_player_weights(
    pd.read_html("https://www.basketball-reference.com/teams/DEN/2024.html")
)

{'Nikola Jokić': 0.3175046554934823,
 'Kentavious Caldwell-Pope': 0.063780260707635,
 'Aaron Gordon': 0.10428305400372438,
 'Michael Porter Jr.': 0.13314711359404097,
 'Jamal Murray': 0.13175046554934822,
 'Reggie Jackson': 0.0940409683426443,
 'Christian Braun': 0.05819366852886405,
 'Peyton Watson': 0.024208566108007444,
 'DeAndre Jordan': 0.03584729981378026,
 'Justin Holiday': 0.03677839851024208,
 'Julian Strawther': 0.006517690875232774,
 'Zeke Nnaji': 0.010707635009310984,
 'Collin Gillespie': -0.01163873370577281,
 'Jalen Pickett': -0.0004655493482309128,
 'Hunter Tyson': -0.05586592178770949,
 'Braxton Key': -0.020018621973929233,
 'Jay Huff': 0.0712290502793296}

In [ ]:
df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")

In [ ]:
df[0]['Description']

0     Out (Thumb) - The Hawks announced that Bufin's...
1     Out (Back) - The Hawks announced that Guete ha...
2     Day To Day (Quad) - Hunter is questionable for...
3     Out (Wrist) - Johnson is expected to miss 4-6 ...
4     Out (Foot) - Whitehead did not play in Saturda...
                            ...                        
86    Out (Calf) - Davis is out for Monday's (Dec. 1...
87    Out (Hip) - Gafford is out for Monday's (Dec. ...
88    Out (Knee) - Rollins hasn't played since Nov. ...
89    Out (Rib) - Shamet is out for Monday's (Dec. 1...
90    Out (Knee) - Wright is expected to miss 4-6 we...
Name: Description, Length: 91, dtype: object

## Find players that are out (cbssports method)

In [ ]:
# def find_team_name(tag):
#     pattern = r'>(.*?)<\/a>'
#     team_name = str(tag.find_all(class_ = "TeamName")[0])
#     team_name = re.search(pattern, team_name).group(1)
#     team_name = team_name.split('>', 1)[-1]
#     return team_name

In [ ]:
# def find_injury_time(tag):
#     element = str(tag).split("width: 40%;")[1]
#     start_index = element.find('">') + 2
#     end_index = element.find('</td>')
#     injury_time = element[start_index:end_index].strip()
#     if injury_time != "Game Time Decision":
#         return "Out"
#     else:
#         return "Questionable"

In [ ]:
# def find_injuries(teams_element):
#     injuries = {}
#     for team_element in teams_element:
#         team_name = find_team_name(team_element)
#         injuries[team_name] = []
#         for report_element in team_element.find_all('tr', class_='TableBase-bodyTr'):
#             player_name = report_element.find('span', class_='CellPlayerName--long').text
#             # injury_time = find_injury_time(report_element)
#             injuries[team_name].append(player_name)
#     return injuries


In [ ]:
# r = requests.get('https://www.cbssports.com/nba/injuries/')
# soup = BeautifulSoup(r.text, 'html.parser')
# teams = soup.find_all(class_ = 'TableBaseWrapper')

# injuries = find_injuries(teams)